# Q2 · Evolution of fire causes
Six cause groups as shares of annual fires. Hover for counts and annual totals.

**Coverage:** GovEsp, 1998–2022. GenCat records lack cause data. Unknown causes are included in the denominator.

**Grouping:** Agricultural burning (210–219), lightning (100), intentional (400), unknown (500 or missing), rekindling (600), and other recorded causes.


### Annual counts
Years with zero fires for a cause are included.


In [ ]:
import pandas as pd
import altair as alt

fires = pd.read_csv("Clean data/CleanData.csv", parse_dates=["data_incendi"])
years = fires.data_incendi.dt.year
fires = fires.loc[fires.font.eq("GovEsp") & years.between(1998, 2022)].copy()
analysis_years = range(1998, 2023)
fires["year"] = fires.data_incendi.dt.year
cause_names = {
    100: "Lightning",
    400: "Intentional",
    500: "Unknown",
    600: "Rekindling",
}
fires["cause"] = fires.codi_causa.map(cause_names).fillna("Other recorded causes")
fires.loc[fires.codi_causa.between(210, 219), "cause"] = "Agricultural burning"
fires.loc[fires.codi_causa.isna(), "cause"] = "Unknown"
groups = [
    "Intentional",
    "Agricultural burning",
    "Other recorded causes",
    "Lightning",
    "Rekindling",
    "Unknown",
]


def annual_counts(records, category, categories):
    columns = ["year", category]
    all_groups = pd.MultiIndex.from_product([analysis_years, categories], names=columns)
    counts = records.groupby(columns).size().reindex(all_groups, fill_value=0)
    return counts.reset_index(name="fires")


annual = annual_counts(fires, "cause", groups)
annual["total"] = annual.groupby("year").fires.transform("sum")
annual["share"] = annual.fires / annual.total
print(f"{len(fires):,} GovEsp fires; missing GenCat causes are not inferred.")


In [ ]:
chart = alt.Chart(annual).mark_line(strokeWidth=2.5).encode(
    x=alt.X("year:Q", title="Year", scale=alt.Scale(zero=False), axis=alt.Axis(format="d", tickCount=8)),
    y=alt.Y(
        "share:Q",
        title="Share of annual fires",
        axis=alt.Axis(format=".0%"),
        scale=alt.Scale(zero=True),
    ),
    color=alt.Color(
        "cause:N",
        title="Cause group",
        scale=alt.Scale(
            domain=groups,
            range=["#0072b2", "#e69f00", "#009e73", "#cc79a7", "#56b4e9", "#666666"],
        ),
    ),
    strokeDash=alt.StrokeDash("cause:N", title="Cause group", sort=groups),
    tooltip=["cause:N", alt.Tooltip("year:O", title="Year"), alt.Tooltip("share:Q", format=".1%"), "fires:Q", alt.Tooltip("total:Q", title="All fires that year")]
).properties(background="white", width=720, height=370, title="Q2 · How the cause mix changed (GovEsp, 1998–2022)")
chart


## Seven cause groups

| Group | Rule / examples |
|---|---|
| Intentional | 400; also 335, explicitly intentional vehicle burning |
| Unknown | 500 or missing cause code |
| Lightning | 100 |
| Vegetation / waste burning | 210–239, 260–289, 295, 370: agricultural, livestock, forestry, vegetation clearance, waste/landfill and escaped controlled burns |
| Equipment / infrastructure | 310–369, except 335: railways, electricity, machinery, vehicles, military activities and tools |
| Rekindling | 600 |
| Other recorded causes | Remaining codes: smoking, campfires/barbecues, fireworks, miscellaneous activities and unspecified recorded causes |

Lines are labelled directly. Code 335 counts as intentional, and unknown causes remain a separate group.


### Grouping and labels
Assign groups and space the endpoint labels.


In [ ]:
grouped = fires.copy()
cause_codes = grouped.codi_causa
grouped["group"] = "Other recorded causes"

burning = (
    cause_codes.between(210, 239)
    | cause_codes.between(260, 289)
    | cause_codes.isin([295, 370])
)
grouped.loc[burning, "group"] = "Vegetation / waste burning"
grouped.loc[cause_codes.between(310, 369), "group"] = "Equipment / infrastructure"

specific_causes = {100: "Lightning", 335: "Intentional", 400: "Intentional", 500: "Unknown", 600: "Rekindling"}
for code, name in specific_causes.items():
    grouped.loc[cause_codes.eq(code), "group"] = name
grouped.loc[cause_codes.isna(), "group"] = "Unknown"

cause_groups = [
    "Intentional",
    "Unknown",
    "Lightning",
    "Vegetation / waste burning",
    "Equipment / infrastructure",
    "Other recorded causes",
    "Rekindling",
]
trends = annual_counts(grouped, "group", cause_groups)
trends["share"] = trends.fires / trends.groupby("year").fires.transform("sum")


def endpoint_labels(series, year, value="share", gap=0.025):
    ends = series.loc[series.year.eq(year)].sort_values(value).copy()
    positions = []
    for point in ends[value]:
        position = max(point, positions[-1] + gap) if positions else point
        positions.append(position)
    ends["label_share"] = positions
    return ends


ends = endpoint_labels(trends, 2022)


In [ ]:
base = alt.Chart(trends).encode(
    x=alt.X(
        "year:Q",
        title="Year",
        scale=alt.Scale(domain=[1998, 2023]),
        axis=alt.Axis(format="d", values=[1998, 2002, 2006, 2010, 2014, 2018, 2022]),
    ),
    y=alt.Y(
        "share:Q",
        title="Percentage of annual fires",
        axis=alt.Axis(format=".0%"),
        scale=alt.Scale(zero=True),
    ),
    color=alt.Color("group:N", legend=None, scale=alt.Scale(domain=cause_groups, scheme="dark2")),
    tooltip=[alt.Tooltip("group:N", title="Cause group"), "year:O", "fires:Q", alt.Tooltip(
        "share:Q",
        title="Percentage",
        format=".1%",
    )])
moving_year_axis = alt.X(
    "year:Q", title="Year",
    scale=alt.Scale(domain=[1999, 2021], nice=False, clamp=False),
    axis=alt.Axis(format="d", values=[1999, 2003, 2007, 2011, 2015, 2019, 2021]),
)

def labelled_trend(chart, ends, label_year, axis_title):
    lines = chart.mark_line(strokeWidth=2.5)
    connectors = chart.mark_rule(strokeWidth=1).encode(
        x2=alt.datum(label_year), y2="label_share:Q",
    ).properties(data=ends)
    labels = chart.mark_text(align="left", dx=6, fontSize=14).encode(
        x=alt.datum(label_year), y=alt.Y("label_share:Q", title=axis_title), text="group:N",
    ).properties(data=ends)
    return (lines + connectors + labels).configure_axis(titleFontSize=14, labelFontSize=12)


group_chart = labelled_trend(base, ends, 2023, "Percentage of annual fires").properties(
    background="white",
    width=700,
    height=420,
    title="Q2 · Cause groups over time (GovEsp, 1998–2022)",
)
group_chart


In [ ]:
smoothed = trends.sort_values(["group", "year"]).copy()
smoothed["share"] = smoothed.groupby("group").share.transform(lambda s: s.rolling(3, center=True, min_periods=3).mean())
smoothed = smoothed.dropna(subset=["share"])
smoothed["start"], smoothed["end"] = smoothed.year - 1, smoothed.year + 1
smooth_ends = endpoint_labels(smoothed, 2021)


## Absolute fire counts
Each point averages three years of fire counts for its cause group.


In [ ]:
smoothed_counts = trends.sort_values(["group", "year"]).copy()
smoothed_counts["fires"] = smoothed_counts.groupby("group").fires.transform(
    lambda s: s.rolling(3, center=True, min_periods=3).mean()
)
smoothed_counts = smoothed_counts.dropna(subset=["fires"])
smoothed_counts["start"], smoothed_counts["end"] = smoothed_counts.year - 1, smoothed_counts.year + 1
count_max = float(trends.fires.max())
count_ends = endpoint_labels(smoothed_counts, 2021, value="fires", gap=0.025 * count_max)
count_axis = "3-year mean of annual fire counts"
count_base = base.properties(data=smoothed_counts).encode(
    x=moving_year_axis,
    y=alt.Y("fires:Q", title=count_axis, axis=alt.Axis(format=",.0f"),
            scale=alt.Scale(domain=[0, max(count_max, float(count_ends.label_share.max()))], nice=True)),
    tooltip=[alt.Tooltip("group:N", title="Cause group"),
             alt.Tooltip("year:O", title="Center year"),
             alt.Tooltip("start:O", title="Window starts"),
             alt.Tooltip("end:O", title="Window ends"),
             alt.Tooltip("fires:Q", title="3-year mean fire count", format=",.1f")])
count_chart = labelled_trend(count_base, count_ends, 2022, count_axis).properties(
    background="white", width=700, height=420, title="Q2 · Centered 3-year moving average of fire counts (1999–2021)")
count_chart


## Final View
Each point averages the annual percentages from the previous, current, and following year. For example, 2010 averages 2009–2011.

Complete windows cover **1999–2021**. The seven groups sum to 100% each year. Smoothing reduces short-lived peaks.


In [ ]:
smooth_base = base.properties(data=smoothed).encode(
    x=moving_year_axis,
    y=alt.Y("share:Q", title="3-year mean of annual percentages", 
            axis=alt.Axis(format=".0%"), 
            scale=alt.Scale(domain=[0, max(trends.share.max(), ends.label_share.max())], 
                            nice=True)),
    tooltip=[alt.Tooltip("group:N", title="Cause group"), 
             alt.Tooltip("year:O", title="Center year"), 
             alt.Tooltip("start:O", title="Window starts"), 
             alt.Tooltip("end:O", title="Window ends"), 
             alt.Tooltip("share:Q", title="3-year mean", format=".1%")])
moving_chart = (labelled_trend(
    smooth_base, smooth_ends, 2022, "3-year mean of annual percentages",
).properties(background="white", width=700, height=420, title="Centered 3-year moving average of the relative frequency of different causes (1999–2021)")
 .configure_title(fontSize=18, subtitleFontSize=12)
 .configure_legend(titleFontSize=14, labelFontSize=12)
)
moving_chart


## View explanation


We chose a line chart because it makes trends and changes in the relative importance of fire causes easy to follow.

To avoid too many lines, we grouped similar causes into seven categories. These are intentional, unknown, lightning, vegetation/waste burning, equipment/infrastructure, rekindling, and other recorded causes. This keeps the chart readable while preserving meaningful differences. Every eligible fire is counted, although grouping hides some detail.

We initially plotted absolute counts, but the overall decline in fires dominated, distracting from changes in causes. Annual percentages make these changes easier to compare. However, a rising percentage does not necessarily mean more fires.

The final view uses a centered three-year moving average. Each point averages the previous, current and following year’s percentages. Three years reduces yearly fluctuations while keeping changes that a five-year average could hide. Centering keeps trends aligned with their time rather than shifting them later. However, smoothing can hide brief peaks.

We use a colourblind-friendly palette and write each group’s name beside its line, so readers can identify it without checking a legend. Lines can still overlap. The chart ends in 2021 because GenCat lacks cause data and the centered average needs the following year’s GovEsp data.